# Harry Potter RAG — Lab 3: Evaluation

Continuing from [Lab 2: Evaluation Prep](./lab2-evaluation-prep.ipynb), which produced
the labelled set at `labs/eval/qa_candidates.jsonl`.

This lab measures the pipeline from Lab 1: it loads the labelled questions, runs the
retrieval and generation stages, and puts numbers behind every design choice.

*Disclaimer: The source book is copyrighted and is not included in this repository. This lab is for personal, non-commercial study only — only original code, questions, and metrics are published. AI coding agent is used in the process to assist notes writing and project planning.*

## Goal of Lab 3: RAG Evaluation & Ablation Matrix

We take the baseline pipeline and measure how different design choices actually move
the numbers. Instead of treating RAG like a black box, we split evaluation into two
separate questions:

1. Did we retrieve the right context?
2. Did the LLM generate a correct, grounded answer?

### What we're testing

1. **Ground-truth evaluation** — load the labelled Q&A set: questions with exact
   evidence snippets, multi-chapter questions, and out-of-scope queries that test
   refusal behaviour.

2. **Ablation grid** — an automated matrix comparison across the main knobs:
    - **Chunking:** fixed-size (256 vs. 512 tokens) vs. chapter-aware boundaries.
    - **Retrieval:** dense (Titan V2 embeddings), sparse (BM25) and hybrid (Reciprocal Rank Fusion).

3. **Separating the failures**
    - **Retriever metrics:** `Hit@k`, `Recall@k` and `MRR` — did the labelled evidence make the context window?
    - **Generator metrics:** an LLM-as-a-Judge for answer correctness and faithfulness, plus false-abstention rate.

4. **Results leaderboard** — every run logs to `eval/results.csv`, so architectural
   choices are backed by numbers rather than vibes.

## Preconditions

- **Lab 1 pipeline** — embeddings, the Chroma collection, BM25, and the `ask()` loop.
- **Lab 2 output** — `labs/eval/qa_candidates.jsonl`. Filter on `locatable` and treat
  `evidence_chunk_ids` as the retrieval label. Lab 3 never regenerates it.

## Build order

1. Load the labelled set and the chunk index.
2. Build the config matrix: chunk size x chunking strategy x retriever.
3. Retrieve, then score `Hit@k` / `Recall@k` / `MRR` against `evidence_chunk_ids`.
4. Score generation: `EM` / `F1`, plus a judge for correctness and faithfulness.
5. Emit one row per question to `labs/eval/results.csv`.

## Import dependencies

In [1]:
import boto3
from botocore.config import Config
import pymupdf
import re
import json
import os
import hashlib
from pathlib import Path
import tiktoken
import numpy as np
from tqdm.auto import tqdm
import chromadb
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
import csv
import collections

/Users/tc/Documents/Github/monitus_code/labs/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Reusing functions from previous lab

In [2]:
def repo_root() -> Path:
    """Walk up from the cwd until the repo root (the dir containing .git)."""
    p = Path.cwd()
    while p != p.parent and not (p / ".git").exists():
        p = p.parent
    return p

ROOT = repo_root()
LABS = ROOT / "labs"
DATA = LABS / "data"

load_dotenv(dotenv_path=str(LABS / ".env"))

class Bedrock:
    """Bedrock helpers: Converse for chat, Titan for embeddings (sequential + cached)."""

    def __init__(self, model_id=None, region=None, embedding_model_id=None):
        self.region = region or os.getenv("AWS_REGION") or "ap-northeast-1"
        self.model_id = model_id or os.getenv("BEDROCK_CONVERSATION_MODEL_ID")
        if not self.model_id:
            raise ValueError("Set BEDROCK_CONVERSATION_MODEL_ID in .env")
        self.embedding_model_id = embedding_model_id or os.getenv("BEDROCK_EMBEDDING_MODEL_ID")
        if not self.embedding_model_id:
            raise ValueError("Set BEDROCK_EMBEDDING_MODEL_ID in .env")
        self.client = boto3.client(
            "bedrock-runtime",
            region_name=self.region,
            config=Config(read_timeout=120, retries={"max_attempts": 5, "mode": "standard"}),
        )

    # ---- conversation (Q&A and judge): override model/params per call ----
    def chat(self, prompt, system=None, model_id=None, temperature=0.0, max_tokens=1024):
        kwargs = {
            "modelId": model_id or self.model_id,
            "messages": [{"role": "user", "content": [{"text": prompt}]}],
            "inferenceConfig": {"temperature": temperature, "maxTokens": max_tokens},
        }
        if system:
            kwargs["system"] = [{"text": system}]
        response = self.client.converse(**kwargs)
        return response["output"]["message"]["content"][0]["text"]

    # ---- embeddings: sequential + disk cache ----
    @staticmethod
    def _cache_path():
        return DATA / "embeddings_cache.jsonl"

    @staticmethod
    def _cache_key(text, model_id, dimensions, normalize):
        raw = f"{model_id}|{dimensions}|{normalize}|{text}"
        return hashlib.sha256(raw.encode("utf-8")).hexdigest()

    def _embed_one(self, text, dimensions, normalize):
        body = json.dumps({"inputText": text, "dimensions": dimensions, "normalize": normalize})
        resp = self.client.invoke_model(modelId=self.embedding_model_id, body=body)
        return json.loads(resp["body"].read())["embedding"]

    def embed(self, texts, dimensions=1024, normalize=True, cache_path=None):
        """Embed texts one by one, appending results to a JSONL cache.

        Returns an (n, dimensions) float32 array. 
        Re-running the notebook will proceed search for already-cached texts, 
        those will make no API calls.
        """
        if isinstance(texts, str):
            texts = [texts]
        texts = list(texts)
        cache_path = Path(cache_path) if cache_path else self._cache_path()

        cache = {}
        if cache_path.exists():
            for line in cache_path.read_text(encoding="utf-8").splitlines():
                if line.strip():
                    rec = json.loads(line)
                    cache[rec["key"]] = rec["vector"]

        model = self.embedding_model_id
        keys = [self._cache_key(t, model, dimensions, normalize) for t in texts]
        todo = [(i, t) for i, t in enumerate(texts) if keys[i] not in cache]

        if todo:
            cache_path.parent.mkdir(parents=True, exist_ok=True)
            with open(cache_path, "a", encoding="utf-8") as f:

                # Process bar
                for i, t in tqdm(todo, desc="embedding"):
                    vec = self._embed_one(t, dimensions, normalize)
                    cache[keys[i]] = vec
                    f.write(json.dumps({"key": keys[i], "vector": vec}) + "\n")

        return np.array([cache[k] for k in keys], dtype=np.float32)


bedrock_client = Bedrock()
judge_client = Bedrock(model_id=os.getenv("BEDROCK_JUDGE_MODEL_ID"))

chroma_client = chromadb.PersistentClient(path=str(LABS / ".chroma"))

## Load the evaluation set

In [3]:
# Labelled evaluation set produced by Lab 2 (Evaluation Prep).

def repo_root() -> Path:
    p = Path.cwd()
    while p != p.parent and not (p / ".git").exists():
        p = p.parent
    return p


EVAL = repo_root() / "labs" / "eval"
rows = [json.loads(line) for line in open(EVAL / "qa_candidates.jsonl", encoding="utf-8") if line.strip()]
usable = [r for r in rows if r.get("locatable")]

print(f"{len(rows)} candidates | {len(usable)} usable (locatable)")
print("types:", dict(collections.Counter(r["type"] for r in usable)))
print("chapters:", len({r["chapter"] for r in usable}))
usable[0]

69 candidates | 65 usable (locatable)
types: {'factoid': 62, 'multi-hop': 3}
chapters: 17


{'id': 'cand-hp1-01-01',
 'book': 1,
 'chapter': 1,
 'chapter_name': 'Chapter 1 - The Boy Who Lived',
 'type': 'factoid',
 'question': 'What is the name of the firm where Mr. Dursley works?',
 'answer': 'Grunnings',
 'evidence': 'Mr. Dursley was the director of a firm called Grunnings, which made drills.',
 'evidence_chunk_ids': ['hp1-ch01-0001'],
 'n_spans': 1,
 'locatable': True,
 'distinctive': True,
 'unmatched_spans': [],
 'status': 'draft'}

## Rebuild the retrieval schema

Lab 3 runs on its own rather than importing from Lab 1. 

The chunks, the Chroma collection and the BM25 index are re-attached here

Therefore, the collection name and `INDEX_CONFIG` must match the ones used at ingest, because the retrieval labels in `qa_candidates.jsonl` are ids from that index.

In [4]:
INDEX_CONFIG = {
    "embedding_model": "amazon.titan-embed-text-v2:0",
    "dimensions": 1024,
    "normalize": True,
    "chunk_size": 512,
    "overlap_ratio": 0.15,
    "tokenizer": "o200k_base",
    "chunking": "sliding_window",
}


class BedrockOnly:
    """Sentinel embedding function: every vector must come from Bedrock."""

    def __call__(self, input):
        raise RuntimeError("Chroma must not embed; pass embeddings= from Bedrock.")

    def embed_query(self, input):
        raise RuntimeError("Use query_embeddings= with Bedrock vectors.")

    def name(self):
        return "bedrock_only"


def get_collection(client, name, config=INDEX_CONFIG):
    client.get_or_create_collection(
        name=name,
        embedding_function=BedrockOnly(),
        configuration={"hnsw": {"space": "cosine"}},   # distance = 1 - cos_sim
        metadata=config,
    )
    return client.get_collection(name, embedding_function=BedrockOnly())


COLLECTION_NAME = "hp1-c512-ov15"
collection = get_collection(chroma_client, name=COLLECTION_NAME)
collection.count()

249

## Load the chunk index and build the lexical index

Lexical index: "maps individual words, terms, or tokens directly to their locations within a document"

Chroma: has vector + metadata + id

BM25 is built on raw chunk + metadata

In [5]:
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


CHUNKS = read_jsonl(DATA / "harrypotter-chunks.jsonl")
CHUNK_BY_ID = {c["chunk_id"]: c for c in CHUNKS}

# Match length between chunk ids vs actual chunk to avoid mismatch
assert len(CHUNK_BY_ID) == len(CHUNKS), "duplicate chunk ids in chunk index"
assert collection.count() == len(CHUNKS), (
    f"collection has {collection.count()} vectors but index has {len(CHUNKS)} chunks"
)


def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())


# Lexical index over the exact same chunks that were embedded
bm25 = BM25Okapi([tokenize(c["text"]) for c in CHUNKS])

print(f"{len(CHUNKS)} chunks | {collection.count()} vectors | {len(bm25.idf)} BM25 terms")

249 chunks | 249 vectors | 5737 BM25 terms


## Three retrievers behind one interface

All three return ordered chunk ids. 

Scoring (and ranking) is done on **ids**, not by direct strings.

### Reciprocal Rank Fusion (RRF)

The hybrid retriever fuses two rankings whose scores are not comparable: dense
returns a **cosine distance** (0-2, smaller is better) and BM25 returns an
unbounded **term-weight sum** (larger is better). Adding them or normalising them
is fragile. RRF sidesteps the problem by discarding the scores and using only
**rank position**:

$$
\mathrm{score}(d) = \sum_{r \in \{\text{dense},\ \text{sparse}\}} \frac{1}{k + \mathrm{rank}_r(d)}
$$

Every chunk receives a contribution from each list it appears in, so a chunk
retrieved by **both** retrievers accumulates credit. The top-k of the fused
ranking is returned.

- `rrf_k = 60` (the damping constant, from Cormack et al. 2009) flattens the
  curve: ranks 1, 2 and 3 contribute `1/61`, `1/62` and `1/63`, so a single
  rank-1 hit cannot dominate, while agreement across both lists still wins.
- This is **not** the same `k` as `top_k`. `top_k` controls how many results are
  returned; `rrf_k` is a fixed constant and belongs in the repro record as the
  **fusion constant**.

Worked example from the smoke run below:

| chunk | dense rank | sparse rank | fused score (`rrf_k = 60`) |
|---|---|---|---|
| `hp1-ch01-0001` | 1 | 1 | `2/61 = 0.0328` |
| `hp1-ch01-0003` | 2 | 3 | `1/62 + 1/63 = 0.0320` |
| `hp1-ch01-0006` | 3 | 2 | `1/63 + 1/62 = 0.0320` |

`hp1-ch01-0001` wins because both retrievers agree on it; the other two tie and
keep insertion order.

In [6]:
def dense_ranking(question_embedding, k):
    """Cosine top-k from the Chroma collection."""
    return collection.query(
        query_embeddings=question_embedding,
        n_results=k,
        include=[],
    )["ids"][0]


def sparse_ranking(question, k):
    """BM25 top-k; argsort maps a score position back to its chunk."""
    scores = bm25.get_scores(tokenize(question))
    return [CHUNKS[i]["chunk_id"] for i in np.argsort(scores)[::-1][:k]]


def hybrid_ranking(question_embedding, question, k, rrf_k=60):
    """Reciprocal Rank Fusion: score = sum(1 / (rrf_k + rank)) over both rankings."""
    fused = {}
    for ranking in (dense_ranking(question_embedding, k), sparse_ranking(question, k)):
        for rank, chunk_id in enumerate(ranking, start=1):
            fused[chunk_id] = fused.get(chunk_id, 0.0) + 1.0 / (rrf_k + rank)
    return [cid for cid, _ in sorted(fused.items(), key=lambda kv: kv[1], reverse=True)[:k]]



| Retriever | Uses | Ranking basis |
|---|---|---|
| `dense` | `embedding` only | cosine distance between query vector and chunk vectors |
| `sparse` | `question` only | BM25 term overlap after `tokenize(question)` |
| `hybrid` | both | RRF over the two rankings |


In [7]:
# Uniform call shape: (question, question_embedding, k) -> ranked chunk ids
RETRIEVERS = {
    "dense": lambda question, embedding, k: dense_ranking(embedding, k),
    "sparse": lambda question, embedding, k: sparse_ranking(question, k),
    "hybrid": lambda question, embedding, k: hybrid_ranking(embedding, question, k),
}

smoke = usable[0]
for name, fn in RETRIEVERS.items():
    print(f"{name:>6}: {fn(smoke['question'], bedrock_client.embed(smoke['question']), 3)}")
print("expected evidence:", smoke["evidence_chunk_ids"])

 dense: ['hp1-ch01-0001', 'hp1-ch01-0003', 'hp1-ch01-0006']
sparse: ['hp1-ch01-0001', 'hp1-ch01-0006', 'hp1-ch01-0003']


hybrid: ['hp1-ch01-0001', 'hp1-ch01-0003', 'hp1-ch01-0006']
expected evidence: ['hp1-ch01-0001']


## Score retrieval against the evidence labels

`Hit@k`, `Recall@k` and `MRR` are computed per question, then averaged. This is a
pure retrieval measurement — no generation is involved yet.

- `Hit@k` — how often at least one labelled chunk made the window.
- `Recall@k` — how much of the labelled evidence made it; 13 of the 65 questions
  carry more than one relevant chunk, so this can be below `Hit@k`.
- `MRR` — how high the first relevant chunk sat.

`top_k` and the fusion constant (`rrf_k = 60`) are repro knobs: record them with
any number you publish. Next comes the config matrix (chunk size / chunking
strategy, which requires rebuilding indices and re-locating the evidence spans),
then the generation half — `EM` / `F1`, the judge, and abstention.

In [8]:
def retrieval_scores(retrieved_ids, relevant_ids):
    """Hit@k, Recall@k and reciprocal rank for one question."""
    relevant = set(relevant_ids)
    retrieved = list(retrieved_ids)
    hits = relevant.intersection(retrieved)
    rr = next(
        (1.0 / rank for rank, cid in enumerate(retrieved, start=1) if cid in relevant),
        0.0,
    )
    return {
        "hit": 1.0 if hits else 0.0,
        "recall": len(hits) / len(relevant) if relevant else 0.0,
        "rr": rr,
    }


def evaluate_retrieval(questions, k=5, retrievers=RETRIEVERS):
    """Score every retriever on the same questions and return mean metrics."""
    questions = list(questions)

    # Embed every question once, in one cached batch, and reuse the vector for
    # both dense and hybrid. The disk cache makes a re-run API-free.
    embeddings = bedrock_client.embed([r["question"] for r in questions])

    scores = {name: [] for name in retrievers}
    for record, embedding in zip(tqdm(questions, desc=f"retrieval@{k}"), embeddings):
        for name, fn in retrievers.items():
            scores[name].append(
                retrieval_scores(
                    fn(record["question"], embedding.reshape(1, -1), k),
                    record["evidence_chunk_ids"],
                )
            )

    table = []
    for name, rows_ in scores.items():
        n = len(rows_)
        table.append({
            "retriever": name,
            f"hit@{k}": sum(r["hit"] for r in rows_) / n,
            f"recall@{k}": sum(r["recall"] for r in rows_) / n,
            "mrr": sum(r["rr"] for r in rows_) / n,
        })
    return table


K = 5
results = evaluate_retrieval(usable, k=K)

header = f"{'retriever':<8}" + "".join(f"{col:>10}" for col in results[0] if col != "retriever")
print(header)
for row in results:
    print(f"{row['retriever']:<8}" + "".join(f"{row[col]:>10.3f}" for col in row if col != "retriever"))

retrieval@5:   0%|          | 0/65 [00:00<?, ?it/s]

retrieval@5:  71%|███████   | 46/65 [00:00<00:00, 456.38it/s]

retrieval@5: 100%|██████████| 65/65 [00:00<00:00, 457.40it/s]

retriever     hit@5  recall@5       mrr
dense        0.862     0.846     0.680
sparse       0.862     0.854     0.697
hybrid       0.908     0.900     0.742


### What the three scores say

| retriever | Hit@5 | Recall@5 | MRR |
|---|---|---|---|
| dense | 0.862 | 0.846 | 0.680 |
| sparse | 0.862 | 0.854 | 0.697 |
| hybrid | **0.908** | **0.900** | **0.742** |

- Hybrid wins on all three: Hit@5 rises to 59/65 (+4.6 points) and MRR to
  0.742, versus 0.680 (dense) and 0.697 (sparse): its first relevant chunk is
  ranked higher, not merely present. Where one retriever misses, the other often
  still has the chunk, and RRF promotes it.
- However, retrieval is not solved, roughly 9% of questions still have no labelled
  chunk in the top 5, and Recall@5 of 0.90 means a tenth of the evidence never
  reached the prompt. 
- That is the headroom the next milestones attack. First with chunking and retriever  ablations, then by separating retrieval misses from generation failures.

These are baseline numbers: one index (512-token chunks, 15% overlap), one
`top_k = 5`, and the Lab 2 label set. They are a fixed point to compare against,
not a final ranking of the retrievers.

## Retrieval failure analysis

Metrics say *how often* retrieval fails; this section says *why*. It takes every
question the hybrid retriever misses and asks three questions about it:

1. Where did the gold chunk actually rank in dense and in sparse, across the
   whole book?
2. Would it have been found if search were restricted to the question's own
   chapter? (A diagnostic, not a deployable filter - inference does not know the
   chapter.)
3. Does widening the fusion pool recover it?

In [9]:
def fuse(rankings, k, rrf_k=60):
    """RRF over precomputed rankings - the body of hybrid_ranking()."""
    fused = {}
    for ranking in rankings:
        for rank, chunk_id in enumerate(ranking, start=1):
            fused[chunk_id] = fused.get(chunk_id, 0.0) + 1.0 / (rrf_k + rank)
    return [cid for cid, _ in sorted(fused.items(), key=lambda kv: kv[1], reverse=True)[:k]]


# Dense side of the diagnostic, computed from the stored vectors rather than a
# deep collection.query(). Requesting n_results equal to the collection size can
# intermittently return one fewer id (Chroma HNSW), which silently drops a chunk
# and makes a deep-pool sweep flip between runs.
_stored = collection.get(include=["embeddings"])
DENSE_IDS = list(_stored["ids"])
DENSE_MAT = np.asarray(_stored["embeddings"], dtype=np.float32)
DENSE_MAT /= np.linalg.norm(DENSE_MAT, axis=1, keepdims=True)
assert len(DENSE_IDS) == DENSE_MAT.shape[0] == len(CHUNKS)


def full_rankings(question):
    """Deterministic full-book dense and sparse rankings, for diagnosis only."""
    query = bedrock_client.embed(question).reshape(-1).astype(np.float32)
    query /= np.linalg.norm(query)
    dense = [DENSE_IDS[i] for i in np.argsort(-(DENSE_MAT @ query))]
    scores = bm25.get_scores(tokenize(question))
    sparse = [CHUNKS[i]["chunk_id"] for i in np.argsort(scores)[::-1]]
    return dense, sparse


def first_rank(ranking, relevant):
    """1-based rank of the best relevant chunk, or None if absent."""
    return next((i for i, cid in enumerate(ranking, start=1) if cid in relevant), None)


# Full rankings are computed once and reused by every diagnostic below.
FULL = {r["id"]: full_rankings(r["question"]) for r in tqdm(usable, desc="full rankings")}

K = 5
failures = []
for record in usable:
    relevant = set(record["evidence_chunk_ids"])
    dense, sparse = FULL[record["id"]]
    retrieved = fuse([dense[:K], sparse[:K]], K)          # identical to hybrid_ranking()
    if relevant & set(retrieved):
        continue

    dense_ch = [c for c in dense if CHUNK_BY_ID[c]["chapter_num"] == record["chapter"]]
    sparse_ch = [c for c in sparse if CHUNK_BY_ID[c]["chapter_num"] == record["chapter"]]
    failures.append({
        "id": record["id"],
        "chapter": record["chapter"],
        "question": record["question"],
        "gold": sorted(relevant),
        "dense_rank": first_rank(dense, relevant),
        "sparse_rank": first_rank(sparse, relevant),
        "in_chapter_hit": bool(relevant & set(fuse([dense_ch[:K], sparse_ch[:K]], K))),
        "retrieved": retrieved,
    })

print(f"{len(failures)} hybrid misses out of {len(usable)}\n")
for f in failures:
    print(f"{f['id']} ch{f['chapter']}: {f['question']}")
    print(f"  gold {f['gold']}")
    print(f"  dense rank {f['dense_rank']} | sparse rank {f['sparse_rank']}"
          f" | recovered inside chapter: {f['in_chapter_hit']}")
    print(f"  retrieved: {f['retrieved']}")

recovered = sum(f["in_chapter_hit"] for f in failures)
chapter_hits = (len(usable) - len(failures)) + recovered
print(f"\nchapter-scoped hybrid Hit@{K}: {chapter_hits}/{len(usable)}"
      f" (global: {len(usable) - len(failures)}/{len(usable)})")

print("\nfusion pool depth vs global Hit@5:")
for pool in (K, 20, len(CHUNKS)):
    hits = sum(
        bool(set(r["evidence_chunk_ids"]) & set(fuse([FULL[r["id"]][0][:pool],
                                                      FULL[r["id"]][1][:pool]], K)))
        for r in usable
    )
    print(f"  pool={pool:>3}: {hits}/{len(usable)}")

full rankings:   0%|          | 0/65 [00:00<?, ?it/s]

full rankings:   2%|▏         | 1/65 [00:00<00:08,  7.28it/s]

full rankings:   5%|▍         | 3/65 [00:00<00:06, 10.30it/s]

full rankings:   8%|▊         | 5/65 [00:00<00:05, 11.23it/s]

full rankings:  11%|█         | 7/65 [00:00<00:05, 11.44it/s]

...

full rankings:  94%|█████████▍| 61/65 [00:05<00:00, 12.30it/s]

full rankings:  97%|█████████▋| 63/65 [00:05<00:00, 12.34it/s]

full rankings: 100%|██████████| 65/65 [00:05<00:00, 12.29it/s]

full rankings: 100%|██████████| 65/65 [00:05<00:00, 11.83it/s]

6 hybrid misses out of 65

cand-hp1-01-04 ch1: What distinctive feature did Harry Potter have on his forehead?
  gold ['hp1-ch01-0013']
  dense rank 24 | sparse rank 27 | recovered inside chapter: True
  retrieved: ['hp1-ch15-0013', 'hp1-ch06-0008', 'hp1-ch17-0005', 'hp1-ch06-0009', 'hp1-ch05-0018']
cand-hp1-06-01 ch6: What name did Harry give to his new owl?
  gold ['hp1-ch06-0001']
  dense rank 59 | sparse rank 4 | recovered inside chapter: True
  retrieved: ['hp1-ch06-0010', 'hp1-ch05-0001', 'hp1-ch05-0017', 'hp1-ch04-0005', 'hp1-ch07-0002']
cand-hp1-10-03 ch10: What spell did Ron use to levitate the troll's club?
  gold ['hp1-ch10-0007']
  dense rank 12 | sparse rank 125 | recovered inside chapter: False
  retrieved: ['hp1-ch10-0011', 'hp1-ch10-0010', 'hp1-ch10-0009', 'hp1-ch16-0018', 'hp1-ch10-0012']
cand-hp1-11-03 ch11: What was the score of Harry's first Quidditch match?
  gold ['hp1-ch11-0010']
  dense rank 16 | sparse rank 67 | recovered inside chapter: False
  retrieved: ['hp

### What the six misses have in common

| id | question | gold chunk(s) | dense rank | sparse rank | recovered inside chapter? |
|---|---|---|---|---|---|
| `cand-hp1-01-04` | distinctive feature on Harry's forehead | `hp1-ch01-0013` | 24 | 27 | yes |
| `cand-hp1-06-01` | name of Harry's new owl | `hp1-ch06-0001` | 59 | 4 | yes |
| `cand-hp1-10-03` | spell Ron used to levitate the club | `hp1-ch10-0007` | 12 | 125 | **no** |
| `cand-hp1-11-03` | score of Harry's first match | `hp1-ch11-0010` | 16 | 67 | **no** |
| `cand-hp1-13-03` | spell Malfoy used on Neville | `hp1-ch13-0002` | 12 | 39 | yes |
| `cand-hp1-16-01` | object used to calm Fluffy | `hp1-ch16-0008`, `0009` | 18 | 7 | yes |

All six are `factoid`, and none is a label error in the "chunk does not contain
the answer" sense - each gold chunk does contain it. Three causes:

**1. Cross-chapter dilution (4 of 6).** Scoping retrieval to the question's own
chapter lifts Hit@5 from 59/65 to **63/65 (0.969)**. The gold chunk ranks fine
*within* its chapter; chunks from other chapters crowd it out of the global top 5.
That is not deployable as-is (inference has no chapter label), but it localises
the loss at book scale rather than in local context, which is the argument for a
reranker over a candidate set rather than a better embedding.

**2. Two genuine in-chapter failures.** In both, a single sentence carries the
answer and the question paraphrases it:

- `cand-hp1-10-03` - the gold chunk is the Charms lesson where the incantation is
  taught, not the troll fight the question describes. The spell is never named in
  the troll scene, so this is partly a **Lab 2 label artifact**: the auto-located
  evidence points at the concept's introduction, not the event.
- `cand-hp1-11-03` - "score" versus "won by one hundred and seventy points to
  sixty": no shared term, and the sentence sits two-thirds into the last chunk of
  the chapter.

**3. Dilution, not distance, is the shared symptom.** The answer is mid-to-late
inside a 512-token chunk (up to 83% through `hp1-ch13-0002`), and four of the six
still retrieve chunks from the *correct* chapter - the right scene, wrong
sentence. In `ch10` and `ch13` the gold sits within two chunk positions of what
was returned. That is the strongest evidence in this notebook for **smaller
chunks (256) plus neighbour expansion**, and it should be the first chunking
ablation.

Two secondary observations:

- **Deeper fusion does not help.** Fusing 249 candidates per list instead of 5
  recovers `cand-hp1-16-01` but breaks `cand-hp1-16-02` and `cand-hp1-17-01`
  (58/65): latent candidates accumulate RRF credit from many weak mentions. Keep
  the pool at `top_k` and fix the chunks instead.
- **RRF can lose a sparse-only win.** `cand-hp1-06-01` is sparse rank 4 but dense
  rank 59, so it lands at fused rank 7 while sparse alone would have hit. Fusion
  trades these away for the questions where dense rescues sparse; it is not
  strictly dominant.

Finally, one partial-recall case: `cand-hp1-09-02` (broomstick models) spans
`hp1-ch09-0007` and `hp1-ch09-0008`, but only the first is retrieved. That is why
`Recall@5` (0.900) trails `Hit@5` (0.908).